# Limit Order Book Alpha & Execution: End-to-End Pipeline (Single Seed)

A walkthrough of the research pipeline on **one synthetic session** (seed 42). The data is
simulated; the generator embeds a hidden informed-trader state, so predictability here
reflects that construction, not a real market. Multi-seed results are in the main README.

1. **Signals**: Level-1 OFI, multi-level OFI with PCA, volume-clock VPIN, micro-price, depth imbalances.
2. **Models**: honest momentum baseline, logistic regression, random forest, LightGBM, DeepLOB, TLOB,
   all with balanced class weights and scored on the same purged test rows.
3. **Validation**: purged CV with embargo, target-permutation sanity check.
4. **Execution**: fantasy mid-price fills vs aggressive taker vs passive maker with trade-through fills.
5. **Submission**: probabilities and actions for the test window.


In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef

notebook_dir = Path.cwd()
project_root = notebook_dir.parent if notebook_dir.name == "submission" else notebook_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.experiments.common import generate_session, prepare_data, SEQ_LEN
from src.data.lob_dataset import build_indexed_dataloaders
from src.models.baselines import NaiveMomentumBaseline, NaiveFlatBaseline, LogisticRegressionBaseline
from src.models.tabular_trees import RandomForestLOB, LightGBMLOB
from src.models.deeplob import DeepLOB
from src.models.tlob import TLOB
from src.models.registry import DeepLearningTrainer, balanced_class_weights
from src.utils.validation_checks import OverfittingDiagnostics
from src.execution.simulator import ExecutionSimulator
from src.execution.strategies import MicrostructureStrategyRunner
from src.execution.metrics import PerformanceMetrics

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"[+] PyTorch device: {'cuda' if torch.cuda.is_available() else 'cpu'}")


[+] PyTorch device: cuda


## 1. Market Data Simulation
60,000 events of a large-tick book (1-tick spread, 10 levels, ~10 ms between events).


In [2]:
session = generate_session(SEED, n_events=60000)
k = 20
y_all = session.labels_dict[k]
lob_40 = session.lob_40
mid_prices = (lob_40[:, 0] + lob_40[:, 2]) / 2.0

print(f"[+] {len(lob_40):,} snapshots over {session.timestamps_ms[-1] / 1000:.0f} s of simulated time")
print(f"[+] Mid-price ${mid_prices[0]:.2f} -> ${mid_prices[-1]:.2f}")
print(f"[+] Class distribution (k={k}):", dict(zip(*np.unique(y_all, return_counts=True))))


[+] 60,000 snapshots over 602 s of simulated time
[+] Mid-price $100.00 -> $97.72
[+] Class distribution (k=20): {np.int64(0): np.int64(40619), np.int64(1): np.int64(9076), np.int64(2): np.int64(10305)}


## 2. Features and Purged Splits
The pipeline (PCA, normalizer) is fit on the training period only; every feature is causal.
Splits are 70/15/15 chronological with a purge gap of k rows between them.


In [3]:
data = prepare_data(session)
idx = data.splits(k)
X = data.tab_feat
X_train, y_train = X[idx["train"]], y_all[idx["train"]]
X_val, y_val = X[idx["val"]], y_all[idx["val"]]
X_test, y_test = X[idx["test"]], y_all[idx["test"]]
print(f"[+] {len(data.feat_names)} features: {data.feat_names[:6]} ...")
print(f"[+] Train {len(X_train):,} | Val {len(X_val):,} | Test {len(X_test):,}")


[+] 27 features: ['imbalance_l1', 'imbalance_l3', 'imbalance_l5', 'imbalance_l10', 'spread_ticks', 'micro_deviation'] ...
[+] Train 41,881 | Val 8,980 | Test 8,980


## 3. Model Hierarchy
The momentum baseline repeats the last *realized* k-event move (label at t-k), the most
recent label actually known at decision time.


In [4]:
results = []
def record(tier, name, preds):
    results.append({"Tier": tier, "Model": name,
                    "Accuracy": accuracy_score(y_test, preds),
                    "Macro F1": f1_score(y_test, preds, average="macro", zero_division=0),
                    "MCC": matthews_corrcoef(y_test, preds)})

record("Tier 0", "Naive Momentum (lag k)", NaiveMomentumBaseline(k).predict(y_all, idx["test"]))
record("Tier 0", "Naive Flat", NaiveFlatBaseline().predict(X_test))

lr_model = LogisticRegressionBaseline(C=1.0); lr_model.fit(X_train, y_train)
record("Tier 1", "Logistic Regression", lr_model.predict(X_test))

rf_model = RandomForestLOB(n_estimators=100, max_depth=10, random_state=SEED); rf_model.fit(X_train, y_train)
record("Tier 2", "Random Forest", rf_model.predict(X_test))

lgb_model = LightGBMLOB(num_leaves=63, max_depth=8, learning_rate=0.05, n_estimators=200, random_state=SEED)
lgb_model.fit(X_train, y_train, X_val, y_val, early_stopping_rounds=15)
preds_lgb = lgb_model.predict(X_test)
probs_lgb = lgb_model.predict_proba(X_test)
record("Tier 3", "LightGBM", preds_lgb)

loaders = build_indexed_dataloaders(data.deep_feat, y_all, idx, seq_len=SEQ_LEN, batch_size=128)
class_w = balanced_class_weights(y_train)
for name, net in [("DeepLOB", DeepLOB(num_classes=3, lstm_hidden=64)),
                  ("TLOB", TLOB(num_features=40, seq_len=SEQ_LEN, d_model=64, num_heads=4, num_layers=2))]:
    trainer = DeepLearningTrainer(net, learning_rate=0.0005, class_weights=class_w)
    trainer.fit(loaders["train"], loaders["val"], epochs=8, patience=3)
    _, _, _, preds, _ = trainer.evaluate(loaders["test"])
    record("Tier 5", name, preds)

display(pd.DataFrame(results))


,Tier,Model,Accuracy,Macro F1,MCC
0,Tier 0,Naive Momentum (lag k),0.619265,0.519147,0.238388
1,Tier 0,Naive Flat,0.666481,0.266622,0.000000
2,Tier 1,Logistic Regression,0.593541,0.580042,0.395440
3,Tier 2,Random Forest,0.625390,0.604957,0.412789
4,Tier 3,LightGBM,0.638641,0.614720,0.425421
5,Tier 5,DeepLOB,0.646214,0.604994,0.397931
6,Tier 5,TLOB,0.534076,0.501368,0.253164


## 4. Validation
* **Purged CV with embargo**: fold-to-fold stability of out-of-sample skill.
* **Target permutation**: training on shuffled targets must give no edge. This catches
  label/row misalignment bugs; it cannot prove the absence of lookahead in features.


In [5]:
df_cv = OverfittingDiagnostics.run_purged_cv_evaluation(LightGBMLOB, X, y_all, n_splits=5, horizon_k=k, embargo_pct=0.01)
display(df_cv)
print(f"Purged CV Macro-F1: {df_cv['test_macro_f1'].mean():.4f} +/- {df_cv['test_macro_f1'].std():.4f}")

perm = OverfittingDiagnostics.run_permutation_reality_check(LightGBMLOB, X_train, y_train, X_test, y_test, n_permutations=3)
print(f"Permuted-target Macro-F1 {perm['permuted_macro_f1']:.4f}, MCC {perm['permuted_mcc']:.4f}",
      "-> FAIL" if perm["leakage_detected"] else "-> PASS")


,fold,train_samples,test_samples,test_accuracy,test_macro_f1,test_mcc
0,1,47380,12000,0.604667,0.593020,0.410323
1,2,47380,12000,0.600750,0.585741,0.398321
2,3,47380,12000,0.609333,0.580911,0.386185
3,4,47980,12000,0.606917,0.591953,0.414657


Purged CV Macro-F1: 0.5879 +/- 0.0057


Permuted-target Macro-F1 0.2788, MCC -0.0379 -> PASS


## 5. Execution Simulation (LightGBM signals, 1 ms latency)
* **Academic**: fills at mid, no fees (upper bound, not achievable).
* **Aggressive taker**: crosses the spread, walks the book, pays 1.5 bps.
* **Passive maker**: joins the touch, fills only when the market trades through the order
  (so fills are adversely selected), earns a 0.5 bps rebate, stands down when VPIN exceeds
  its 90th training percentile.


In [6]:
test_idx = idx["test"]
test_lob = lob_40[test_idx]
test_ts = session.timestamps_ms[test_idx]
vpin = X[:, data.feat_names.index("vpin")]
vpin_thresh = float(np.quantile(vpin[:data.train_end], 0.90))
test_vpin = vpin[test_idx]

sim = ExecutionSimulator(initial_cash=1000000.0, maker_fee_bps=-0.5, taker_fee_bps=1.5,
                         lot_size=100.0, latency_ms=1.0, timestamps_ms=test_ts)
runner = MicrostructureStrategyRunner(sim)
runs = {
    "Academic Mid-Price (Fantasy)": runner.run_academic_midprice(preds_lgb, (test_lob[:, 0] + test_lob[:, 2]) / 2.0),
    "Aggressive Taker": runner.run_realistic_aggressive(preds_lgb, test_lob),
    "Passive Maker (VPIN gate)": runner.run_dynamic_passive_vpin(preds_lgb, test_lob, test_vpin, vpin_thresh),
}
rows = []
for name, res in runs.items():
    perf = PerformanceMetrics.calculate_metrics(res["equity"], 1000000.0, timestamps_ms=test_ts, bar_ms=100.0)
    rows.append({"Strategy": name, "Net PnL ($)": perf["net_pnl"], "Trades": res["num_trades"],
                 "Fees ($)": res.get("total_fees", 0.0), "Slippage ($)": res.get("total_slippage", 0.0)})
print(f"Test window: {(test_ts[-1] - test_ts[0]) / 1000:.0f} s of simulated trading")
display(pd.DataFrame(rows))


Test window: 90 s of simulated trading


,Strategy,Net PnL ($),Trades,Fees ($),Slippage ($)
0,Academic Mid-Price (Fantasy),1987.000000,831,0.000000,0.000000
1,Aggressive Taker,253.342604,831,1214.335587,524.321808
2,Passive Maker (VPIN gate),23.882350,9,-4.382350,0.000000


## 6. Submission File
Test-window probabilities, predicted labels, VPIN, and the action the passive maker would take.


In [7]:
submission_df = pd.DataFrame({
    "event_id": test_idx,
    "prob_flat": probs_lgb[:, 0],
    "prob_up": probs_lgb[:, 1],
    "prob_down": probs_lgb[:, 2],
    "predicted_label": preds_lgb,
    "vpin_toxicity": test_vpin,
})
submission_df["recommended_action"] = np.select(
    [submission_df["vpin_toxicity"] > vpin_thresh, submission_df["predicted_label"] == 1, submission_df["predicted_label"] == 2],
    ["CANCEL_VPIN_TOXIC", "POST_PASSIVE_BID", "POST_PASSIVE_ASK"],
    default="HOLD_FLAT",
)
sub_path = Path("submission.csv") if Path.cwd().name == "submission" else Path("submission/submission.csv")
submission_df.to_csv(sub_path, index=False)
print(f"[+] Saved {len(submission_df):,} rows to {sub_path} (VPIN toxic threshold = {vpin_thresh:.3f})")
display(submission_df.head(10))


[+] Saved 8,980 rows to submission\submission.csv (VPIN toxic threshold = 0.740)


,event_id,prob_flat,prob_up,prob_down,predicted_label,vpin_toxicity,recommended_action
0,51000,0.645217,0.312395,0.042387,0,0.698173,HOLD_FLAT
1,51001,0.579329,0.353652,0.067020,0,0.698173,HOLD_FLAT
2,51002,0.664213,0.239443,0.096344,0,0.698173,HOLD_FLAT
3,51003,0.740506,0.155109,0.104385,0,0.698173,HOLD_FLAT
4,51004,0.483532,0.272982,0.243486,0,0.698173,HOLD_FLAT
5,51005,0.601796,0.128756,0.269448,0,0.698173,HOLD_FLAT
6,51006,0.446143,0.046282,0.507575,2,0.698173,POST_PASSIVE_ASK
7,51007,0.377047,0.034060,0.588893,2,0.698173,POST_PASSIVE_ASK
8,51008,0.377043,0.036173,0.586785,2,0.698173,POST_PASSIVE_ASK
9,51009,0.565694,0.193916,0.240390,0,0.698173,HOLD_FLAT
